In [1]:
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

In [2]:
SUPPORTED_STOCKS = [
    "AAPL",
    "MSFT",
    "GOOGL",
    "AMZN",
    "NVDA"
]

In [3]:
def calculate_rsi(close, period=14):

    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss

    rsi = 100 - (100/(1+rs))

    return rsi

In [4]:
for ticker in SUPPORTED_STOCKS:

    print("="*70)
    print(f"Processing {ticker}")
    print("="*70)

    RAW_DIR = f"../data/{ticker}/raw"
    PROCESSED_DIR = f"../data/{ticker}/processed"

    os.makedirs(PROCESSED_DIR, exist_ok=True)

    stock = pd.read_csv(f"{RAW_DIR}/yahoo_stock.csv")

    company = pd.read_csv(f"{RAW_DIR}/company_info.csv")

    print(stock.shape)

Processing AAPL
(894, 7)
Processing MSFT
(894, 7)
Processing GOOGL
(894, 7)
Processing AMZN
(894, 7)
Processing NVDA
(894, 7)


In [21]:
def preprocess_stock(ticker):

    print("\n" + "=" * 70)
    print(f"Processing {ticker}")
    print("=" * 70)

    # ==========================================================
    # Define Paths
    # ==========================================================
    RAW_DIR = f"../data/{ticker}/raw"
    PROCESSED_DIR = f"../data/{ticker}/processed"

    os.makedirs(PROCESSED_DIR, exist_ok=True)

    # ==========================================================
    # Load Datasets
    # ==========================================================
    stock = pd.read_csv(f"{RAW_DIR}/yahoo_stock.csv")
    company = pd.read_csv(f"{RAW_DIR}/company_info.csv")

    print(f"Raw Stock Dataset : {stock.shape}")
    print(f"Company Dataset   : {company.shape}")

    # ==========================================================
    # Date Conversion
    # ==========================================================
    stock["Date"] = pd.to_datetime(stock["Date"])

    stock = stock.sort_values("Date").reset_index(drop=True)

    # ==========================================================
    # Merge Company Information
    # ==========================================================
    for col in company.columns:
        if col not in stock.columns:
            stock[col] = company.iloc[0][col]

    print("After Merge :", stock.shape)

    # ==========================================================
    # Feature Engineering
    # ==========================================================

    # Simple Moving Averages
    stock["SMA10"] = stock["Close"].rolling(10).mean()
    stock["SMA20"] = stock["Close"].rolling(20).mean()
    stock["SMA50"] = stock["Close"].rolling(50).mean()

    # Exponential Moving Averages
    stock["EMA10"] = stock["Close"].ewm(span=10, adjust=False).mean()
    stock["EMA20"] = stock["Close"].ewm(span=20, adjust=False).mean()

    # RSI
    stock["RSI"] = calculate_rsi(stock["Close"])

    # MACD
    ema12 = stock["Close"].ewm(span=12, adjust=False).mean()
    ema26 = stock["Close"].ewm(span=26, adjust=False).mean()

    stock["MACD"] = ema12 - ema26
    stock["MACD_Signal"] = stock["MACD"].ewm(span=9, adjust=False).mean()
    stock["MACD_Hist"] = stock["MACD"] - stock["MACD_Signal"]

    # Bollinger Bands
    stock["BB_Middle"] = stock["Close"].rolling(20).mean()

    rolling_std = stock["Close"].rolling(20).std()

    stock["BB_Upper"] = stock["BB_Middle"] + (2 * rolling_std)
    stock["BB_Lower"] = stock["BB_Middle"] - (2 * rolling_std)

    # Returns
    stock["Daily_Return"] = stock["Close"].pct_change()

    stock["Log_Return"] = np.log(
        stock["Close"] / stock["Close"].shift(1)
    )

    stock["Volatility"] = (
        stock["Daily_Return"]
        .rolling(20)
        .std()
    )

    # Lag Features
    stock["Lag1"] = stock["Close"].shift(1)
    stock["Lag2"] = stock["Close"].shift(2)
    stock["Lag3"] = stock["Close"].shift(3)

    print("Feature Engineering Completed")
    print("Current Shape :", stock.shape)

    # ==========================================================
    # Handle Missing Dividend Information
    # ==========================================================

    dividend_columns = [
        "DividendPerShare",
        "DividendYield",
        "DividendDate",
        "ExDividendDate"
    ]

    for col in dividend_columns:
        if col in stock.columns:
            if stock[col].dtype == "object":
                stock[col] = stock[col].fillna("Not Available")
            else:
                stock[col] = stock[col].fillna(0)

    # ==========================================================
    # Data Cleaning
    # ==========================================================

    required_columns = [
        "Close",
        "SMA10",
        "SMA20",
        "SMA50",
        "EMA10",
        "EMA20",
        "RSI",
        "MACD",
        "MACD_Signal",
        "MACD_Hist",
        "BB_Middle",
        "BB_Upper",
        "BB_Lower",
        "Daily_Return",
        "Log_Return",
        "Volatility",
        "Lag1",
        "Lag2",
        "Lag3"
    ]

    print("\nRows before cleaning :", len(stock))
    print("\nNaN Count in Required Columns")
    print(stock[required_columns].isnull().sum())

    stock = stock.dropna(subset=required_columns).reset_index(drop=True)

    print("\nAfter Cleaning")
    print("Rows    :", stock.shape[0])
    print("Columns :", stock.shape[1])

    # ==========================================================
    # Validation
    # ==========================================================

    print("\nValidation Report")
    print("-" * 30)
    print("Missing Values :", stock.isnull().sum().sum())
    print("Duplicate Rows :", stock.duplicated().sum())

    print("\nDate Range")
    print(stock["Date"].min(), "to", stock["Date"].max())

    # ==========================================================
    # Save Processed Dataset
    # ==========================================================

    output_file = os.path.join(
        PROCESSED_DIR,
        "processed_stock_data.csv"
    )

    stock.to_csv(output_file, index=False)

    print("\nProcessed dataset saved successfully!")
    print("Location :", output_file)
    print("Final Shape :", stock.shape)

In [22]:
for ticker in SUPPORTED_STOCKS:
    preprocess_stock(ticker)

print("\n" + "="*70)
print("LAB 4 COMPLETED SUCCESSFULLY")
print("="*70)


Processing AAPL
Raw Stock Dataset : (894, 7)
Company Dataset   : (1, 55)
After Merge : (894, 62)
Feature Engineering Completed
Current Shape : (894, 80)

Rows before cleaning : 894

NaN Count in Required Columns
Close            0
SMA10            9
SMA20           19
SMA50           49
EMA10            0
EMA20            0
RSI             14
MACD             0
MACD_Signal      0
MACD_Hist        0
BB_Middle       19
BB_Upper        19
BB_Lower        19
Daily_Return     1
Log_Return       1
Volatility      20
Lag1             1
Lag2             2
Lag3             3
dtype: int64

After Cleaning
Rows    : 845
Columns : 80

Validation Report
------------------------------
Missing Values : 0
Duplicate Rows : 0

Date Range
2023-03-15 00:00:00 to 2026-07-28 00:00:00

Processed dataset saved successfully!
Location : ../data/AAPL/processed\processed_stock_data.csv
Final Shape : (845, 80)

Processing MSFT
Raw Stock Dataset : (894, 7)
Company Dataset   : (1, 55)
After Merge : (894, 62)
Feature